# Notebook 05 — Quality extensions (README §5)

Extra experiments for the content-based bonus criteria. Modules 1–4 and their outputs are **not modified**.

| Criterion | What this notebook adds |
|---|---|
| Experimentation | null model and stability for K-Means, X-Means, ablations (scale, outliers, attributes), binning sensitivity, regression ablation |
| Performance | 12 engineered attributes, tunable prior correction τ, nested CV |
| Innovation | paired bootstrap, ordinal analysis, learning curves, model agreement, Simpson's paradox (size vs dealer type), train→test generalization of the rules |
| Appearance | `m5_*` figures and tables used by the LaTeX report |

**Discipline.** Every choice is made by CV on the Module 3 training split (same folds). The test split is read once more, for one configuration per model fixed by CV.

**Outside the lectures.** Random forest and gradient boosting are used only as a yardstick (never selected). As README §2 says, mention them to the instructor beforehand, or drop them from the report.

**How to run.** With `RECOMPUTE = False` (default) the slow stages load the tables of the full run, and the notebook takes seconds. `RECOMPUTE = True` recomputes everything (about 15 minutes of CPU and several GB of RAM); keep `N_JOBS` small on a laptop.

In [1]:
import os
# Safety switches: the full computation needs several GB of RAM and about 15 minutes of CPU. By default the slow stages load
# the tables saved by the full run (tables/m5_*.csv); the light stages (tree, size effect, Mantel-Haenszel, figures) always run.
RECOMPUTE = False
N_JOBS = 2          # parallel workers when RECOMPUTE is True
for v_ in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS", "LOKY_MAX_CPU_COUNT"]:
    os.environ.setdefault(v_, "1" if v_ != "LOKY_MAX_CPU_COUNT" else "2")
import sys, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SRC = str(ROOT / "src")
sys.path.insert(0, SRC)
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")   # parallel workers import m5_utils
import plot_style as ps
from m3_utils import PriorCorrectedKNN
from m5_utils import (Prep, PriorPower, TauTree, OrdinalTree, BinnedNB, RuleClassifier, ENGINEERED, xmeans,
                      fit_encoder, encode_items)

from joblib import Parallel, delayed
from mlxtend.frequent_patterns import apriori, association_rules
from scipy import stats
from sklearn.base import clone
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (accuracy_score, f1_score, silhouette_score, adjusted_rand_score, cohen_kappa_score,
                             roc_auc_score, confusion_matrix, recall_score, r2_score, mean_squared_error)
from sklearn.model_selection import (StratifiedKFold, KFold, GridSearchCV, cross_val_predict, train_test_split)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

ps.setup()
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 110)
SEED = 42
CLASSES = ps.CLASS_ORDER

df = pd.read_csv(ROOT / "data/processed/clean_base.csv", index_col="sp_id")
fs = json.load(open(ROOT / "data/processed/feature_sets.json"))
split = pd.read_csv(ROOT / "data/processed/split.csv", index_col="sp_id")["set"].loc[df.index]
clusters = pd.read_csv(ROOT / "data/processed/cluster_labels.csv", index_col="sp_id").loc[df.index]
TARGET, NUM, CAT, LOGV = fs["target"], fs["numeric_model"], fs["categorical_model"], fs["log1p_vars"]

# The split is re-created in the row order of Module 3, so that folds and tie-breaks are identical to Module 3
tr_idx, te_idx = train_test_split(df.index, test_size=0.2, stratify=df[TARGET], random_state=SEED)
assert set(tr_idx) == set(df.index[split == "train"])
dtr, dte = df.loc[tr_idx], df.loc[te_idx]
ytr, yte = dtr[TARGET].to_numpy(), dte[TARGET].to_numpy()
CV = StratifiedKFold(5, shuffle=True, random_state=SEED)   # the Module 3 folds
FOLDS = list(CV.split(dtr, ytr))
print(dtr.shape, dte.shape)

def prep(**kw):
    return Prep(NUM, CAT, LOGV, **kw)

def fold_f1(est, a, b):
    return f1_score(ytr[b], clone(est).fit(dtr.iloc[a], ytr[a]).predict(dtr.iloc[b]), average="macro")

def cv_f1(est, folds=FOLDS):
    """Macro-F1 per fold of a pipeline that receives the raw data frame (all statistics fit on the fold's training part)."""
    return np.array(Parallel(n_jobs=N_JOBS)(delayed(fold_f1)(est, a, b) for a, b in folds))

def saved(name, **kw):
    """Table written by the full run (RECOMPUTE = True)."""
    return pd.read_csv(ps.TAB_DIR / f"{name}.csv", **kw)

(11074, 31) (2769, 31)


## 1. Cluster validity (Module 2)

Three questions left open by Module 2: is the K-Means structure stronger than in structureless data, is k = 4 reproducible, and what does an automatic criterion say?

- **Null model:** 20 datasets with each column permuted independently (same marginals, no dependence), clustered in the same way.
- **Stability:** 20 pairs of 80% subsamples per k; ARI on the common sellers.
- **X-Means** (Pelleg & Moore): clusters are split while the BIC of two spherical Gaussians beats one, up to a cap of 64.

In [2]:
KS = range(2, 9)
if RECOMPUTE:
    F = ["n_listings", "price_median", "savings_amount_median", "daysonmarket_median", "mileage_median", "pct_is_new", "listing_span_days"]
    V = df[F].fillna(df[F].median())
    for c in F:
        if c in LOGV:
            V[c] = np.log1p(V[c])
    X7 = StandardScaler().fit_transform(V)
    KS = range(2, 9)

    def km_stats(X, k, n_init=5):
        m = KMeans(k, n_init=n_init, random_state=SEED).fit(X)
        return m.inertia_, silhouette_score(X, m.labels_)

    def null_rep(r):
        rng = np.random.default_rng(1000 + r)
        Xn = np.column_stack([rng.permutation(X7[:, j]) for j in range(X7.shape[1])])   # same marginals, no dependence
        return [(r, k, *km_stats(Xn, k)) for k in KS]

    real = pd.DataFrame([(k, *km_stats(X7, k, n_init=10)) for k in KS], columns=["k", "SSE", "silhouette"]).set_index("k")
    null = pd.DataFrame(sum(Parallel(n_jobs=N_JOBS)(delayed(null_rep)(r) for r in range(20)), []), columns=["rep", "k", "SSE", "silhouette"])
    nv = real.join(null.groupby("k").agg(null_SSE=("SSE", "mean"), null_SSE_sd=("SSE", "std"), null_sil=("silhouette", "mean"),
                                         null_sil_sd=("silhouette", "std"), null_sil_max=("silhouette", "max")))
    nv["SSE / null SSE"] = nv.SSE / nv.null_SSE
    nv.to_csv(ps.TAB_DIR / "m5_cluster_null.csv")
else:
    nv = saved("m5_cluster_null", index_col=0)
nv.round(4)

,SSE,silhouette,null_SSE,null_SSE_sd,null_sil,null_sil_sd,null_sil_max,SSE / null SSE
k,,,,,,,,
2,52823.5736,0.4212,83691.6193,274.8192,0.1556,0.0045,0.1581,0.6312
3,42777.4658,0.3334,76064.0188,275.4586,0.1423,0.0011,0.1439,0.5624
4,36131.4872,0.3431,70967.7476,400.2017,0.1406,0.0092,0.1567,0.5091
5,33031.2250,0.3253,66077.4870,344.6469,0.1431,0.0092,0.1621,0.4999
6,30661.5590,0.2369,62130.0165,435.6789,0.1463,0.0090,0.1647,0.4935
7,28484.0268,0.2166,58446.1364,402.5794,0.1462,0.0079,0.1579,0.4874
8,26936.8272,0.2092,55590.2698,468.8626,0.1410,0.0061,0.1494,0.4846


In [3]:
if RECOMPUTE:
    def stab(k, r):
        """ARI between two K-Means fits on two independent 80% subsamples, measured on the sellers common to both."""
        rng = np.random.default_rng(2000 + r)
        a, b = (np.sort(rng.choice(len(X7), int(.8 * len(X7)), replace=False)) for _ in range(2))
        la = KMeans(k, n_init=5, random_state=r).fit(X7[a]).labels_
        lb = KMeans(k, n_init=5, random_state=r + 500).fit(X7[b]).labels_
        com = np.intersect1d(a, b)
        return k, r, adjusted_rand_score(la[np.searchsorted(a, com)], lb[np.searchsorted(b, com)])

    st = pd.DataFrame(Parallel(n_jobs=N_JOBS)(delayed(stab)(k, r) for k in KS for r in range(20)), columns=["k", "rep", "ARI"])
    st_tab = st.groupby("k").ARI.agg(["mean", "std", "min"])
    st_tab.to_csv(ps.TAB_DIR / "m5_cluster_stability.csv")

    xm_lab, xm_hist = xmeans(X7, k_max=64)
    xm = pd.DataFrame(xm_hist, columns=["k", "BIC", "accepted splits"])
    xm.to_csv(ps.TAB_DIR / "m5_xmeans.csv", index=False)
    print("X-Means final k =", len(np.unique(xm_lab)), "| silhouette %.3f" % silhouette_score(X7, xm_lab),
          "| ARI vs K-Means k=4: %.3f" % adjusted_rand_score(xm_lab, clusters.kmeans4))
    print(xm.round(0).to_string(index=False))
else:
    st_tab, xm = saved("m5_cluster_stability", index_col=0), saved("m5_xmeans")
    print(xm.round(0).to_string(index=False))
st_tab.round(3)

 k       BIC  accepted splits
 2 -117661.0                2
 4 -107766.0                4
 8 -102777.0                7
15  -99177.0               12
27  -95281.0               14
41  -92083.0               16
57  -90376.0               20
64  -90625.0               21


,mean,std,min
k,,,
2,0.998,0.001,0.995
3,0.991,0.006,0.980
4,0.992,0.004,0.984
5,0.967,0.068,0.682
6,0.900,0.145,0.610
7,0.955,0.020,0.913
8,0.937,0.046,0.763


In [4]:
fig, axes = plt.subplots(1, 3, figsize=(ps.FULL_W, 2.2))
ax = axes[0]
ax.plot(nv.index, nv.silhouette, marker="o", markersize=4, color=ps.CATEGORICAL[0], label="data")
ax.errorbar(nv.index, nv.null_sil, yerr=2 * nv.null_sil_sd, marker="s", markersize=3, color=ps.NEUTRAL, linestyle="--",
            linewidth=1.2, label="permuted null (mean ± 2 sd)")
ax.set_ylim(0, 0.52); ax.set_title("Silhouette vs null model"); ax.set_ylabel("silhouette"); ax.legend(frameon=False, fontsize=6.5)
ax = axes[1]
ax.plot(nv.index, nv["SSE / null SSE"], marker="o", markersize=4, color=ps.CATEGORICAL[0])
ax.set_title("SSE relative to null"); ax.set_ylabel("SSE / null SSE")
ax = axes[2]
ax.errorbar(st_tab.index, st_tab["mean"], yerr=st_tab["std"], marker="o", markersize=4, color=ps.CATEGORICAL[0], capsize=2)
ax.set_ylim(0.0, 1.05); ax.set_title("Subsample stability"); ax.set_ylabel("ARI (mean ± sd)")
for a in axes:
    a.set_xticks(list(KS)); a.set_xlabel("k"); a.axvline(4, color=ps.NEUTRAL, linestyle=":", linewidth=1)
fig.tight_layout()
ps.save(fig, "m5_cluster_validity")

**Reading.**
- The null silhouette stays at 0.14–0.16 for every k, while the data reach 0.34 at k = 4: the structure is real. The margin falls from 0.20 (k = 4) to 0.07 (k = 8), and the SSE ratio to the null stops improving after k = 4.
- ARI between subsamples is ≥ 0.99 for k ≤ 4, then unstable (k = 5: 0.97 ± 0.07; k = 6: 0.90 ± 0.15). k = 4 is the finest reproducible partition.
- **X-Means does not stop** before the cap: with ≈ 14k sellers the BIC penalty is negligible against any departure from a spherical Gaussian. The data are a continuum that can be refined indefinitely.

## 2. Sensitivity of the Module 3 baselines (ablation)

One element at a time is changed in the configurations chosen in Module 3, on identical folds. `delta` is the mean change of macro-F1, `folds better` counts the folds where the variant wins.

In [5]:
if RECOMPUTE:
    SIZE = ["n_listings"]
    SPECS = ["horsepower_median", "city_fuel_economy_median", "back_legroom_median", "front_legroom_median",
             "fuel_tank_volume_median", "height_median", "length_median", "width_median"]
    HIST = ["pct_isCab", "pct_salvage", "pct_theft_title", "owner_count_median"]
    COMM = ["price_median", "savings_amount_median", "daysonmarket_median", "listing_span_days", "mileage_median", "pct_is_new"]
    assert set(SIZE + SPECS + HIST + COMM) == set(NUM)

    # reference = the configurations chosen in Module 3
    dt_m3 = DecisionTreeClassifier(max_depth=5, min_samples_leaf=50, class_weight="balanced", random_state=SEED)
    knn_m3 = PriorCorrectedKNN(151, p=1, weights="distance", prior_correction=True)
    gnb_m3 = GaussianNB(var_smoothing=0.1, priors=[1 / 3] * 3)
    REF = {"DT": (dt_m3, {}), "KNN": (knn_m3, dict(log=True, scaler="standard")),
           "GNB": (gnb_m3, dict(log=True, scaler="standard", drop=tuple(CAT)))}

    def pipe(model, kw, **over):
        return Pipeline([("prep", prep(**{**kw, **over})), ("clf", clone(model))])

    abl = []
    def add(model, group, variant, scores, ref):
        abl.append({"model": model, "ablation": group, "variant": variant, "CV macro-F1": scores.mean(), "sd": scores.std(),
                    "delta": (scores - ref).mean(), "folds better": int((scores > ref).sum())})

    noise = clusters.dbscan.loc[dtr.index].to_numpy() == -1
    for m, (clf, kw) in REF.items():
        ref = cv_f1(pipe(clf, kw))
        add(m, "reference", "Module 3 configuration", ref, ref)
        if m != "DT":   # trees are invariant to monotone transforms of the features
            for name, o in [("raw, unscaled", dict(log=False, scaler=None)), ("raw, standardized", dict(log=False)),
                            ("log1p, min-max", dict(scaler="minmax")), ("log1p, robust", dict(scaler="robust"))]:
                add(m, "scale", name, cv_f1(pipe(clf, kw, **o)), ref)
        add(m, "outliers", "winsorized at p1/p99 (per fold)", cv_f1(pipe(clf, kw, winsor=True)), ref)
        add(m, "outliers", "DBSCAN noise removed from training folds",
            cv_f1(pipe(clf, kw), folds=[(a[~noise[a]], b) for a, b in FOLDS]), ref)
        base_drop = kw.get("drop", ())
        for name, cols in [("without seller size", SIZE), ("without vehicle specs", SPECS), ("without vehicle history", HIST),
                           ("without price/turnover/wear", COMM)] + ([("without categoricals", CAT)] if m != "GNB" else []):
            add(m, "features removed", name, cv_f1(pipe(clf, kw, drop=tuple(base_drop) + tuple(cols))), ref)
        add(m, "features removed", "only seller size (n_listings)",
            cv_f1(pipe(clf, kw, drop=tuple(c for c in NUM + CAT if c != "n_listings"))), ref)
        add(m, "features added", "12 engineered features", cv_f1(pipe(clf, kw, engineered=True)), ref)
        add(m, "features added", "K-Means segment + centroid distances", cv_f1(pipe(clf, kw, clusters=True)), ref)
    abl = pd.DataFrame(abl)
    abl.to_csv(ps.TAB_DIR / "m5_ablation.csv", index=False)
else:
    abl = saved("m5_ablation")
abl.round(4)

,model,ablation,variant,CV macro-F1,sd,delta,folds better
0,DT,reference,Module 3 configuration,0.4192,0.0138,0.0000,0
1,DT,outliers,winsorized at p1/p99 (per fold),0.4192,0.0138,0.0000,0
2,DT,outliers,DBSCAN noise removed from training folds,0.4012,0.0194,-0.0180,1
3,DT,features removed,without seller size,0.3793,0.0130,-0.0399,0
4,DT,features removed,without vehicle specs,0.4182,0.0148,-0.0010,1
5,DT,features removed,without vehicle history,0.4194,0.0141,0.0002,1
6,DT,features removed,without price/turnover/wear,0.3898,0.0177,-0.0294,0
7,DT,features removed,without categoricals,0.4181,0.0142,-0.0011,0
8,DT,features removed,only seller size (n_listings),0.3577,0.0236,-0.0615,0
9,DT,features added,12 engineered features,0.4169,0.0104,-0.0023,2


**Reading.**
- **Scale** is decisive for distances and for smoothing (KNN −0.063, GaussianNB −0.097 without scaling); the log matters less (−0.003, −0.014).
- **Outliers:** winsorizing changes nothing; removing the DBSCAN noise from the training folds *hurts* every model (−0.010 to −0.018). Keeping the extreme sellers (Module 1) was right.
- **Attributes:** seller size and the price/turnover/wear block are needed; the 8 vehicle specifications add nothing to DT and KNN.
- **Segments as features** help the tree slightly (+0.006, 5/5 folds) and hurt KNN and GaussianNB.

## 3. Extended models: prior-correction exponent τ and engineered attributes

Prediction rule: argmax P(c|x) / P(c)^τ. τ = 0 is the plain rule and τ = 1 the Module 3 prior correction; for the tree the class weights are P(c)^(−τ) (τ = 1 is `balanced`).

The 12 engineered attributes (`ENGINEERED` in `src/m5_utils.py`) use only the seller's own row, so they cannot leak.

In [6]:
TAUS = [0, 0.25, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
KNN_BASE = KNeighborsClassifier(151, p=1, weights="distance", algorithm="brute")
MAKE = {"DT": lambda: Pipeline([("prep", prep()), ("clf", TauTree())]),
        "KNN": lambda: Pipeline([("prep", prep(log=True, scaler="standard")), ("clf", PriorPower(KNN_BASE))]),
        "GNB": lambda: Pipeline([("prep", prep(log=True, scaler="standard", drop=tuple(CAT))), ("clf", PriorPower(GaussianNB(var_smoothing=0.1)))]),
        "CNB": lambda: Pipeline([("prep", prep()), ("clf", BinnedNB())])}
PARAMS = {"DT": {"clf__max_depth": [4, 5, 6, 8, 10], "clf__min_samples_leaf": [20, 50, 100]},
          "KNN": {"clf__base__n_neighbors": [51, 101, 151, 201]},
          "GNB": {"clf__base__var_smoothing": [1e-3, 1e-2, 1e-1]},
          "CNB": {"clf__n_bins": [3, 4, 6]}}

def grid_for(name, taus, eng):
    """List of parameter grids; for CategoricalNB the number of binned columns follows the engineered switch."""
    out = []
    for e in eng:
        g = {"prep__engineered": [e], "clf__tau": taus, **PARAMS[name]}
        if name == "CNB":
            g["clf__n_num"] = [len(NUM) + (len(ENGINEERED) if e else 0)]
        out.append(g)
    return out

def search(name, taus, eng, d, y, cv):
    return GridSearchCV(MAKE[name](), grid_for(name, taus, eng), cv=cv, scoring="f1_macro", n_jobs=N_JOBS, error_score="raise").fit(d, y)

if RECOMPUTE:
    t0 = time.time()
    SEARCH, tau_rows = {}, []
    for name in MAKE:
        s = SEARCH[name] = search(name, TAUS, [False, True], dtr, ytr, CV)
        print(name, len(s.cv_results_["params"]), "configs; best", s.best_params_, "CV macro-F1 %.4f ± %.4f" % (s.best_score_, s.cv_results_["std_test_score"][s.best_index_]), "(%.0fs)" % (time.time() - t0), flush=True)
        r = pd.DataFrame(s.cv_results_)
        for (tau, e), g_ in r.groupby([r.param_clf__tau.astype(float), r.param_prep__engineered.astype(bool)]):
            b = g_.sort_values("mean_test_score").iloc[-1]
            tau_rows.append((name, tau, e, b.mean_test_score, b.std_test_score))
    tau_tab = pd.DataFrame(tau_rows, columns=["model", "tau", "engineered", "best CV macro-F1", "sd"])
    tau_tab.to_csv(ps.TAB_DIR / "m5_tau_grid.csv", index=False)
    pd.concat([pd.DataFrame(s.cv_results_).filter(regex="^param_|mean_test_score|std_test_score").assign(model=n)
               for n, s in SEARCH.items()]).to_csv(ps.TAB_DIR / "m5_classifier_grid.csv", index=False)
else:
    tau_tab = saved("m5_tau_grid")
tau_tab.pivot_table(index=["model", "engineered"], columns="tau", values="best CV macro-F1").round(3)

tau                0.00   0.25   0.50   0.60   0.70   0.80   0.90   1.00
model engineered                                                        
CNB   False       0.412  0.413  0.409  0.407  0.405  0.403  0.401  0.399
      True        0.420  0.418  0.413  0.412  0.408  0.406  0.401  0.395
DT    False       0.402  0.418  0.434  0.442  0.442  0.440  0.437  0.419
      True        0.397  0.422  0.435  0.443  0.445  0.440  0.435  0.418
GNB   False       0.390  0.402  0.413  0.413  0.412  0.410  0.407  0.404
      True        0.423  0.423  0.423  0.422  0.420  0.418  0.418  0.416
KNN   False       0.292  0.344  0.403  0.416  0.433  0.446  0.445  0.434
      True        0.329  0.380  0.426  0.437  0.450  0.456  0.453  0.438

In [7]:
# Nested CV (outer = the 5 folds above, inner = 5 new folds): an honest comparison of whole search spaces
SPACES = [("M3 space (tau 0/1)", [0, 1.0], [False]), ("+ tau grid", TAUS, [False]), ("+ tau grid + engineered", TAUS, [False, True])]
if RECOMPUTE:
    t0 = time.time()
    nested = []
    for i, (a, b) in enumerate(FOLDS):
        for name in MAKE:
            for space, taus, eng in SPACES:
                s = search(name, taus, eng, dtr.iloc[a], ytr[a], StratifiedKFold(5, shuffle=True, random_state=7))
                nested.append((name, space, i, f1_score(ytr[b], s.best_estimator_.predict(dtr.iloc[b]), average="macro"), str(s.best_params_)))
        print("outer fold", i, "%.0fs" % (time.time() - t0), flush=True)
    nested = pd.DataFrame(nested, columns=["model", "space", "fold", "macro-F1", "params"])
    nested.to_csv(ps.TAB_DIR / "m5_nested_folds.csv", index=False)
    nt = nested.groupby(["model", "space"])["macro-F1"].agg(["mean", "std"])
    nw = nested.pivot_table(index=["model", "fold"], columns="space", values="macro-F1")
    for s_ in ["+ tau grid", "+ tau grid + engineered"]:
        nt.loc[(slice(None), s_), "folds better than M3 space"] = (nw[s_] > nw["M3 space (tau 0/1)"]).groupby("model").sum().reindex(nt.loc[(slice(None), s_), :].index.get_level_values(0)).to_numpy()
    nt.to_csv(ps.TAB_DIR / "m5_nested_cv.csv")
else:
    nested = saved("m5_nested_folds")
    nt = saved("m5_nested_cv", index_col=[0, 1])
nt.round(3)

mean    std  folds better than M3 space
model space                                                            
CNB   + tau grid               0.411  0.017                         0.0
      + tau grid + engineered  0.419  0.012                         4.0
      M3 space (tau 0/1)       0.412  0.018                         NaN
DT    + tau grid               0.436  0.008                         5.0
      + tau grid + engineered  0.444  0.007                         5.0
      M3 space (tau 0/1)       0.413  0.012                         NaN
GNB   + tau grid               0.413  0.012                         4.0
      + tau grid + engineered  0.422  0.007                         4.0
      M3 space (tau 0/1)       0.409  0.014                         NaN
KNN   + tau grid               0.446  0.014                         5.0
      + tau grid + engineered  0.452  0.004                         5.0
      M3 space (tau 0/1)       0.432  0.009                         NaN

In [8]:
# Reference ceiling: two ensembles (outside the lecture programme; a yardstick only, never candidates for selection)
if RECOMPUTE:
    ens = {"Random forest (ref.)": Pipeline([("prep", prep()), ("clf", PriorPower(RandomForestClassifier(300, min_samples_leaf=10, random_state=SEED)))]),
           "Gradient boosting (ref.)": Pipeline([("prep", prep()), ("clf", PriorPower(HistGradientBoostingClassifier(learning_rate=.05, max_iter=150, max_leaf_nodes=15, random_state=SEED)))])}
    ENS = {n: GridSearchCV(e, {"clf__tau": TAUS}, cv=CV, scoring="f1_macro", n_jobs=N_JOBS, error_score="raise").fit(dtr, ytr) for n, e in ens.items()}
    print(pd.DataFrame([(n, s.best_params_["clf__tau"], s.best_score_) for n, s in ENS.items()], columns=["model", "tau", "CV macro-F1"]).round(4))

    # Rule-based classifier (Module 4 optimum s=0.005, L=4): tau applied to the class distribution of the selected antecedent
    rb_tau = {t: [] for t in TAUS}
    for a, b in FOLDS:
        m = RuleClassifier(0.005, 4).fit(dtr.iloc[a], TARGET, CLASSES)
        M = m.match(dtr.iloc[b])
        for t in TAUS:
            rb_tau[t].append(f1_score(ytr[b], m.predict(None, t, M), average="macro"))
    rb_tau_tab = pd.Series({t: np.mean(v) for t, v in rb_tau.items()}, name="CV macro-F1")
    rb_tau_tab.rename_axis("tau").to_csv(ps.TAB_DIR / "m5_rule_tau.csv")
    RB_TAU = rb_tau_tab.idxmax()
    print("rule classifier, CV macro-F1 by tau:", rb_tau_tab.round(3).to_dict(), "-> tau =", RB_TAU)
    CEIL = max(s.best_score_ for s in ENS.values())
else:
    CEIL = saved("m5_test_metrics", index_col=0).loc[["Random forest (ref.)", "Gradient boosting (ref.)"], "CV macro-F1"].max()
    rb_tau_tab = saved("m5_rule_tau", index_col=0)["CV macro-F1"]
    print("best ensemble CV macro-F1: %.3f" % CEIL)
    print("rule classifier, CV macro-F1 by tau:", rb_tau_tab.round(3).to_dict())

best ensemble CV macro-F1: 0.462
rule classifier, CV macro-F1 by tau: {0.0: 0.43, 0.25: 0.4, 0.5: 0.342, 0.6: 0.313, 0.7: 0.294, 0.8: 0.283, 0.9: 0.281, 1.0: 0.281}


In [9]:
NICE = {"DT": "Decision Tree", "KNN": "KNN", "CNB": "CategoricalNB", "GNB": "GaussianNB"}
fig, axes = plt.subplots(1, 2, figsize=(ps.FULL_W, 2.4), gridspec_kw={"width_ratios": [1.2, 1]})
ax = axes[0]
for i, n in enumerate(["DT", "KNN", "CNB", "GNB"]):
    g_ = tau_tab[tau_tab.model == n].groupby("tau")["best CV macro-F1"].max()
    ax.plot(g_.index, g_.values, marker="osD^"[i], markersize=3.5, color=ps.CATEGORICAL[i], linestyle=["-", "--", "-.", ":"][i], label=NICE[n], linewidth=1.6)
ax.axhline(CEIL, color=ps.INK_2, linewidth=0.8, linestyle=":")
ax.text(0, CEIL + 0.004, "best ensemble (reference)", fontsize=6.5, color=ps.INK_2)
ax.set_ylim(0.22, 0.49); ax.set_xlabel("prior-correction exponent τ"); ax.set_ylabel("best CV macro-F1")
ax.set_title("Partial prior correction"); ax.legend(frameon=False, loc="lower right")
ax = axes[1]
w = 0.26
for j, (s_, *_rest) in enumerate(SPACES):
    q = nested[nested.space == s_].groupby("model")["macro-F1"].agg(["mean", "std"]).loc[["DT", "KNN", "CNB", "GNB"]]
    ax.errorbar(np.arange(4) + (j - 1) * w, q["mean"], yerr=q["std"], fmt="osD"[j], markersize=4.5, color=["#86b6ef", "#2a78d6", "#0d366b"][j],
                label=["M3 space", "+ τ grid", "+ τ + engineered"][j], elinewidth=1, capsize=2)
ax.set_xticks(range(4)); ax.set_xticklabels(["DT", "KNN", "CNB", "GNB"]); ax.grid(axis="x", visible=False)
ax.set_ylabel("nested-CV macro-F1"); ax.set_title("Nested CV (mean ± sd, 5 outer folds)", pad=16)
ax.legend(frameon=False, ncol=3, loc="lower center", bbox_to_anchor=(0.5, 0.97), fontsize=6.5, columnspacing=0.8, handletextpad=0.2)
fig.tight_layout()
ps.save(fig, "m5_tau")

**Reading.**
- An **intermediate τ** is best for DT (0.7) and KNN (0.8); Naïve Bayes is flat, because its over-confident posteriors are barely moved by a prior factor.
- **Nested CV** (whole search spaces): DT 0.413 → 0.444 and KNN 0.432 → 0.452, better in 5/5 folds. Most of the gain comes from τ; the engineered attributes add 0.006–0.010.
- The rule classifier does not benefit: its best τ is 0 (dividing the already extreme distribution of the selected antecedent by the prior hurts).
- Its CV score here is 0.430 against 0.435 in Module 4, because ties between antecedents of equal lift break differently; the test macro-F1 is 0.433 in both.

## 4. Test set, confidence intervals and paired comparisons

The Module 3 models were refitted first and reproduced the published test scores (0.4369, 0.4328, 0.4072). Intervals: 2,000 bootstrap resamples of the test sellers, the same resamples for all models.

In [10]:
as_rank = lambda s: np.array([CLASSES.index(v) for v in s])
if RECOMPUTE:
    def eval_model(name, pred, proba=None):
        r = {"model": name, "accuracy": accuracy_score(yte, pred), "macro-F1": f1_score(yte, pred, average="macro"),
             "weighted-F1": f1_score(yte, pred, average="weighted"),
             **{f"R {c}": v for c, v in zip(CLASSES, recall_score(yte, pred, labels=CLASSES, average=None))},
             "QWK": cohen_kappa_score(as_rank(yte), as_rank(pred), weights="quadratic"),
             "MAE": np.abs(as_rank(yte) - as_rank(pred)).mean()}
        if proba is not None:
            r["AUC macro"] = roc_auc_score(label_binarize(yte, classes=CLASSES), proba, average="macro")
        return r

    def proba(est):
        return est.predict_proba(dte)[:, [list(est.classes_).index(c) for c in CLASSES]]

    PRED, rows = {}, []
    for n, e in {"DT (M3)": pipe(dt_m3, {}), "KNN (M3)": pipe(knn_m3, REF["KNN"][1]), "GNB (M3)": pipe(gnb_m3, REF["GNB"][1])}.items():
        e.fit(dtr, ytr)
        PRED[n] = e.predict(dte)
        rows.append(eval_model(n, PRED[n], proba(e)))
    # sanity check: the re-fitted Module 3 models must reproduce the published Module 3 numbers
    m3_tab = pd.read_csv(ps.TAB_DIR / "m3_test_metrics.csv", index_col="model")
    for n, k in [("DT (M3)", "Decision Tree"), ("KNN (M3)", "KNN"), ("GNB (M3)", "GaussianNB")]:
        got, want = f1_score(yte, PRED[n], average="macro"), m3_tab.loc[k, "macro-F1"]
        print(f"{n}: macro-F1 {got:.4f} (Module 3 table: {want:.4f})")
        assert abs(got - want) < 2e-3
    for name, s in SEARCH.items():
        n = f"{name} (τ)"
        PRED[n] = s.best_estimator_.predict(dte)
        rows.append(eval_model(n, PRED[n], proba(s.best_estimator_)))
    for n, s in ENS.items():
        PRED[n] = s.predict(dte)
        rows.append(eval_model(n, PRED[n], proba(s.best_estimator_)))
    rb = RuleClassifier(0.005, 4).fit(dtr, TARGET, CLASSES)
    Mte = rb.match(dte)
    PRED["Rules (M4)"] = rb.predict(None, 0.0, Mte)
    rows.append(eval_model("Rules (M4)", PRED["Rules (M4)"]))
    if RB_TAU != 0:
        PRED["Rules (τ)"] = rb.predict(None, RB_TAU, Mte)
        rows.append(eval_model("Rules (τ)", PRED["Rules (τ)"]))
    PRED["Majority"] = np.full(len(yte), "medium")
    rows.append(eval_model("Majority", PRED["Majority"]))
    test = pd.DataFrame(rows).set_index("model")

    # bootstrap over test sellers (2,000 resamples, the same resamples for every model -> paired differences)
    rng = np.random.default_rng(SEED)
    BOOT = [rng.integers(0, len(yte), len(yte)) for _ in range(2000)]
    def fast_f1(t, p):
        cm = np.bincount(t * 3 + p, minlength=9).reshape(3, 3)
        return (2 * np.diag(cm) / np.maximum(cm.sum(0) + cm.sum(1), 1)).mean()
    BF = {n: np.array([fast_f1(as_rank(yte)[i], pr[i]) for i in BOOT]) for n, pr in ((n, as_rank(p)) for n, p in PRED.items())}
    test["F1 lo"] = [np.percentile(BF[n], 2.5) for n in test.index]
    test["F1 hi"] = [np.percentile(BF[n], 97.5) for n in test.index]
    cvs = {**{f"{k} (τ)": s.best_score_ for k, s in SEARCH.items()}, **{k: s.best_score_ for k, s in ENS.items()}, "Rules (τ)": rb_tau_tab.max(), "Rules (M4)": rb_tau_tab[0]}   # 0.430 here vs 0.435 in Module 4: ties between equal-lift antecedents break differently
    m3cv = pd.read_csv(ps.TAB_DIR / "m3_cv_vs_test.csv", index_col=0)["CV macro-F1"]
    cvs.update({"DT (M3)": m3cv["Decision Tree"], "KNN (M3)": m3cv["KNN"], "GNB (M3)": m3cv["GaussianNB"]})
    test.insert(0, "CV macro-F1", [cvs.get(n, np.nan) for n in test.index])
    test.to_csv(ps.TAB_DIR / "m5_test_metrics.csv")
else:
    test = saved("m5_test_metrics", index_col=0)
test.round(3)

,CV macro-F1,accuracy,macro-F1,weighted-F1,R low,R medium,R high,QWK,MAE,AUC macro,F1 lo,F1 hi
model,,,,,,,,,,,,
DT (M3),0.419,0.459,0.437,0.466,0.368,0.430,0.583,0.248,0.623,0.629,0.418,0.455
KNN (M3),0.434,0.449,0.433,0.460,0.457,0.418,0.506,0.209,0.655,0.642,0.415,0.451
GNB (M3),0.404,0.440,0.407,0.452,0.374,0.480,0.402,0.136,0.667,0.593,0.387,0.425
DT (τ),0.445,0.510,0.444,0.502,0.192,0.573,0.604,0.261,0.547,0.634,0.425,0.463
KNN (τ),0.456,0.518,0.464,0.518,0.266,0.577,0.575,0.238,0.557,0.658,0.443,0.483
GNB (τ),0.423,0.471,0.415,0.469,0.202,0.522,0.555,0.198,0.601,0.609,0.396,0.435
CNB (τ),0.420,0.447,0.413,0.449,0.264,0.427,0.617,0.196,0.643,0.610,0.394,0.432
Random forest (ref.),0.462,0.510,0.475,0.515,0.347,0.518,0.608,0.257,0.573,0.668,0.454,0.493
Gradient boosting (ref.),0.457,0.524,0.466,0.521,0.252,0.598,0.563,0.263,0.537,0.664,0.445,0.485


In [11]:
if RECOMPUTE:
    pairs = [("KNN (M3)", "DT (M3)"), ("Rules (M4)", "DT (M3)"), ("GNB (M3)", "DT (M3)"),
             ("DT (τ)", "DT (M3)"), ("KNN (τ)", "KNN (M3)"), ("GNB (τ)", "GNB (M3)"),
             ("KNN (τ)", "DT (τ)"), ("CNB (τ)", "DT (τ)"),
             ("Random forest (ref.)", "DT (τ)"), ("Random forest (ref.)", "KNN (τ)"), ("Gradient boosting (ref.)", "KNN (τ)")]
    pb = pd.DataFrame([(a, b, test.loc[a, "macro-F1"] - test.loc[b, "macro-F1"], *np.percentile(BF[a] - BF[b], [2.5, 97.5])) for a, b in pairs],
                      columns=["model A", "model B", "dF1 (A-B)", "lo", "hi"])
    pb["CI excludes 0"] = (pb.lo > 0) | (pb.hi < 0)
    pb.to_csv(ps.TAB_DIR / "m5_paired_bootstrap.csv", index=False)
else:
    pb = saved("m5_paired_bootstrap")

fig, ax = plt.subplots(figsize=(ps.COL_W, 2.9))
o = test.drop("Majority").sort_values("macro-F1").index
for y_, n in enumerate(o):
    c, mk = (ps.NEUTRAL, "s") if "ref." in n else ((ps.CATEGORICAL[1], "D") if "τ" in n else (ps.CATEGORICAL[0], "o"))
    ax.plot(test.loc[n, ["F1 lo", "F1 hi"]], [y_, y_], color=c, linewidth=1.6)
    ax.plot(test.loc[n, "macro-F1"], y_, marker=mk, color=c, markersize=5)
    ax.text(test["F1 hi"].max() + 0.004, y_, f"{test.loc[n, 'macro-F1']:.3f}", fontsize=7, va="center", color=ps.INK_2)
ax.set_yticks(range(len(o))); ax.set_yticklabels(o)
ax.set_xlim(test.drop("Majority")["F1 lo"].min() - 0.005, test["F1 hi"].max() + 0.022)
ax.set_xlabel("test macro-F1 (95% bootstrap CI)"); ax.grid(axis="y", visible=False)
fig.tight_layout()
ps.save(fig, "m5_test_ci")
pb.round(3)

,model A,model B,dF1 (A-B),lo,hi,CI excludes 0
0,KNN (M3),DT (M3),-0.004,-0.024,0.018,False
1,Rules (M4),DT (M3),-0.004,-0.025,0.017,False
2,GNB (M3),DT (M3),-0.030,-0.052,-0.006,True
3,DT (τ),DT (M3),0.007,-0.011,0.025,False
4,KNN (τ),KNN (M3),0.031,0.013,0.047,True
5,GNB (τ),GNB (M3),0.008,-0.010,0.026,False
6,KNN (τ),DT (τ),0.020,0.000,0.041,True
7,CNB (τ),DT (τ),-0.031,-0.052,-0.008,True
8,Random forest (ref.),DT (τ),0.031,0.012,0.051,True
9,Random forest (ref.),KNN (τ),0.011,-0.004,0.027,False


**Reading.**
- The baseline DT, KNN and rule classifier are statistically tied; the DT beats GaussianNB.
- **Extended KNN** 0.464 [0.443, 0.483] beats baseline KNN (+0.031, CI excludes 0) and is level with both ensembles. The extended tree gains only +0.007 on test (not significant), although +0.026 in CV: the baseline tree had a favourable test draw (CV 0.419, test 0.437).
- **Cost:** the recall of `low` falls (KNN 0.46 → 0.27). The baseline rule remains preferable if the goal is to find low-rated sellers.
- The extended tree (16 leaves) splits first on the **number of used cars**, an engineered attribute. Large leaves keep their precision on the test set; the 47-seller leaf does not (0.70 → 0.38).

In [12]:
# Figure: the extended tree (top 3 levels) with the *unweighted* class mix of the training sellers in each node
# (the configuration is read from the CV grid and the single tree is fitted on the training split: a one-second step)
g_ = saved("m5_classifier_grid")
best = g_[g_.model == "DT"].sort_values("mean_test_score").iloc[-1]
est = Pipeline([("prep", prep(engineered=bool(best.param_prep__engineered))),
                ("clf", TauTree(max_depth=int(best.param_clf__max_depth), min_samples_leaf=int(best.param_clf__min_samples_leaf),
                                tau=float(best.param_clf__tau)))]).fit(dtr, ytr)
print("extended tree:", est["clf"].get_params(), "| test macro-F1 %.4f (table: %.4f)" % (f1_score(yte, est.predict(dte), average="macro"), test.loc["DT (τ)", "macro-F1"]))
assert abs(f1_score(yte, est.predict(dte), average="macro") - test.loc["DT (τ)", "macro-F1"]) < 1e-6
tm, names = est["clf"].tree_model_, est["prep"].get_feature_names()
Xt = est["prep"].transform(dtr)
path = tm.decision_path(Xt).tocsc()
t_ = tm.tree_
print("depth", tm.get_depth(), "leaves", tm.get_n_leaves())
imp = pd.Series(tm.feature_importances_, names).sort_values(ascending=False)
imp[imp > 0].rename("importance").to_csv(ps.TAB_DIR / "m5_tree_importance.csv")
print(imp.head(8).round(3).to_dict())

def cond(node):
    n = names[t_.feature[node]].replace("_median", "").replace("dominant_", "").replace("franchise_status_", "").replace("_amount", "")
    th = t_.threshold[node]
    v = f"{th / 1000:,.1f}k" if abs(th) >= 10000 else (f"{th:,.0f}" if abs(th) >= 100 else f"{th:.1f}" if abs(th) >= 10 else f"{th:.2f}")
    return f"{n}\n≤ {v}"

fig, ax = plt.subplots(figsize=(ps.FULL_W, 2.9))
ax.axis("off"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
YS = [0.90, 0.64, 0.38, 0.11]
MAXD = 3
def draw(node, depth, x, half):
    sel = path[:, node].toarray().ravel() > 0
    mix = pd.Series(ytr[sel]).value_counts(normalize=True).reindex(CLASSES).fillna(0)
    pred = np.array(sorted(CLASSES))[t_.value[node].argmax()]
    leaf = t_.children_left[node] == -1
    txt = f"{sel.sum():,} sellers\n" + " / ".join(f"{100 * v:.0f}" for v in mix)
    if not leaf and depth < MAXD:
        txt = cond(node) + "\n" + txt
    else:
        txt = f"→ {pred}\n" + txt
    if not leaf and depth < MAXD:
        for ch, sg, lb in [(t_.children_left[node], -1, "yes"), (t_.children_right[node], 1, "no")]:
            nx = x + sg * half / 2
            ax.plot([x, nx], [YS[depth] - 0.02, YS[depth + 1] + 0.02], color=ps.NEUTRAL, linewidth=0.8, zorder=0)
            if depth == 0:
                ax.text((x + nx) / 2, (YS[0] + YS[1]) / 2 + 0.05, lb, fontsize=7, color=ps.INK_2, ha="center")
            draw(ch, depth + 1, nx, half / 2)
    ax.text(x, YS[depth], txt, ha="center", va="center", fontsize=6.6, color=ps.INK, linespacing=1.15,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor=ps.CLASS_COLORS[pred], linewidth=1.8))
draw(0, 0, 0.5, 0.5)
ax.legend(handles=[plt.Line2D([], [], color=ps.CLASS_COLORS[c], linewidth=2.5, label=f"predicts {c}") for c in CLASSES], frameon=False, loc="upper right", fontsize=7)
ax.text(0, 0.97, "class mix: low / medium / high (%)", fontsize=7, color=ps.INK_2, va="top")
ps.save(fig, "m5_tree")

# leaves of the full tree with their composition (training and test)
lt, le = tm.apply(Xt), tm.apply(est["prep"].transform(dte))
lv = []
for lf in np.unique(lt):
    cur, parts = lf, []
    while cur != 0:
        par = int(np.where((t_.children_left == cur) | (t_.children_right == cur))[0][0])
        parts.append(cond(par).replace("\n", " ") if t_.children_left[par] == cur else cond(par).replace("\n", " ").replace("≤", ">"))
        cur = par
    tr_m, te_m = pd.Series(ytr[lt == lf]).value_counts(normalize=True), pd.Series(yte[le == lf]).value_counts(normalize=True)
    p_ = np.array(sorted(CLASSES))[t_.value[lf].argmax()]
    lv.append({"prediction": p_, "n train": int((lt == lf).sum()), "precision train": tr_m.get(p_, 0), "precision test": te_m.get(p_, 0), "rule": " AND ".join(parts[::-1])})
lv = pd.DataFrame(lv).sort_values(["prediction", "precision train"], ascending=[True, False])
lv.to_csv(ps.TAB_DIR / "m5_tree_leaves.csv", index=False)
lv.round(3)

extended tree: {'criterion': 'gini', 'max_depth': 4, 'min_samples_leaf': 20, 'random_state': 42, 'tau': 0.7} | test macro-F1 0.4441 (table: 0.4441)
depth 4 leaves 16
{'n_used': 0.629, 'price_median': 0.123, 'n_listings': 0.086, 'savings_ratio': 0.051, 'price_per_hp': 0.034, 'n_new': 0.028, 'franchise_status_franchise': 0.014, 'height_median': 0.011}


,prediction,n train,precision train,precision test,rule
0,high,1402,0.499,0.504,n_used ≤ 45.5 AND n_used ≤ 17.5 AND savings_ratio ≤ 0.10 AND franchise ≤ 0.50
1,high,304,0.447,0.448,n_used ≤ 45.5 AND n_used ≤ 17.5 AND savings_ratio ≤ 0.10 AND franchise > 0.50
5,high,2203,0.385,0.370,n_used ≤ 45.5 AND n_used > 17.5 AND n_new ≤ 116 AND price_per_hp > 53.8
2,high,295,0.383,0.357,n_used ≤ 45.5 AND n_used ≤ 17.5 AND savings_ratio > 0.10 AND active_share ≤ 0.17
7,low,29,0.517,0.300,n_used ≤ 45.5 AND n_used > 17.5 AND n_new > 116 AND daysonmarket > 86.5
10,low,915,0.374,0.348,n_used > 45.5 AND price ≤ 36.2k AND n_listings > 154 AND price ≤ 25.2k
9,low,306,0.330,0.311,n_used > 45.5 AND price ≤ 36.2k AND n_listings ≤ 154 AND savings_ratio > 0.07
14,medium,145,0.876,0.722,n_used > 45.5 AND price > 36.2k AND n_listings > 220 AND height ≤ 67.4
12,medium,94,0.830,0.655,n_used > 45.5 AND price > 36.2k AND n_listings ≤ 220 AND pct_is_new ≤ 0.30
15,medium,167,0.760,0.625,n_used > 45.5 AND price > 36.2k AND n_listings > 220 AND height > 67.4


## 5. Error analysis: ordinality, ceiling, agreement

In [13]:
if RECOMPUTE:
    def adjacency(pred):
        cm = confusion_matrix(as_rank(yte), as_rank(pred), labels=[0, 1, 2]).astype(float)
        exp = np.outer(cm.sum(1), cm.sum(0)) / cm.sum()              # independence of truth and prediction, same marginals
        f = lambda m: (m[0, 2] + m[2, 0]) / (m.sum() - np.trace(m))
        return f(cm), f(exp)

    adj = pd.DataFrame({n: adjacency(PRED[n]) for n in test.index if n != "Majority"}, index=["low-high share of errors", "expected by chance"]).T
    adj["ratio"] = adj.iloc[:, 0] / adj.iloc[:, 1]
    adj = adj.join(test[["QWK", "MAE"]])
    adj.to_csv(ps.TAB_DIR / "m5_ordinal.csv")

    ord_s = GridSearchCV(Pipeline([("prep", prep()), ("clf", OrdinalTree())]),
                         {"prep__engineered": [False, True], "clf__max_depth": [4, 5, 6, 8], "clf__min_samples_leaf": [20, 50, 100], "clf__tau": TAUS},
                         cv=CV, scoring="f1_macro", n_jobs=N_JOBS, error_score="raise").fit(dtr, ytr)
    def oof_scores(est):
        p = cross_val_predict(est, dtr, ytr, cv=CV, n_jobs=N_JOBS)
        return cohen_kappa_score(as_rank(ytr), as_rank(p), weights="quadratic"), f1_score(ytr, p, average="macro"), np.abs(as_rank(ytr) - as_rank(p)).mean()
    oc = pd.DataFrame({"nominal tree (tau)": oof_scores(SEARCH["DT"].best_estimator_), "ordinal tree (Frank-Hall, tau)": oof_scores(ord_s.best_estimator_)},
                      index=["OOF QWK", "OOF macro-F1", "OOF MAE"]).T
    oc.to_csv(ps.TAB_DIR / "m5_ordinal_tree.csv")
    print("ordinal tree:", ord_s.best_params_, round(ord_s.best_score_, 4))
    print(oc.round(4))
else:
    adj, oc = saved("m5_ordinal", index_col=0), saved("m5_ordinal_tree", index_col=0)
    print(oc.round(4))
adj.round(3)

                                OOF QWK  OOF macro-F1  OOF MAE
nominal tree (tau)               0.2329        0.4452   0.5787
ordinal tree (Frank-Hall, tau)   0.2347        0.4476   0.5794


,low-high share of errors,expected by chance,ratio,QWK,MAE
DT (M3),0.153,0.208,0.735,0.248,0.623
KNN (M3),0.187,0.219,0.856,0.209,0.655
GNB (M3),0.190,0.202,0.941,0.136,0.667
DT (τ),0.117,0.166,0.705,0.261,0.547
KNN (τ),0.155,0.175,0.885,0.238,0.557
GNB (τ),0.137,0.176,0.781,0.198,0.601
CNB (τ),0.163,0.201,0.815,0.196,0.643
Random forest (ref.),0.169,0.194,0.875,0.257,0.573
Gradient boosting (ref.),0.130,0.167,0.779,0.263,0.537
Rules (M4),0.143,0.169,0.847,0.195,0.581


In [14]:
if RECOMPUTE:
    # Learning curve on the fixed folds: does more data help?
    def lc(name, est, frac, rep):
        out = []
        for a, b in FOLDS:
            if frac < 1:
                a, _ = train_test_split(a, train_size=frac, stratify=ytr[a], random_state=rep)
            out.append(fold_f1(est, a, b))
        return name, frac, rep, np.mean(out)

    FR = [0.1, 0.2, 0.4, 0.6, 0.8, 1.0]
    jobs = [(n, SEARCH[n].best_estimator_, f, r) for n in ["DT", "KNN"] for f in FR for r in range(3 if f < 1 else 1)]
    L = pd.DataFrame(Parallel(n_jobs=N_JOBS)(delayed(lc)(*j) for j in jobs), columns=["model", "fraction", "rep", "f1"])
    lc_tab = L.groupby(["model", "fraction"]).f1.agg(["mean", "std"]).reset_index()
    lc_tab["n"] = (lc_tab.fraction * len(FOLDS[0][0])).astype(int)
    lc_tab.to_csv(ps.TAB_DIR / "m5_learning_curve.csv", index=False)
else:
    lc_tab = saved("m5_learning_curve")
lc_tab.round(4)

,model,fraction,mean,std,n
0,DT,0.1,0.4099,0.0040,885
1,DT,0.2,0.4169,0.0050,1771
2,DT,0.4,0.4289,0.0054,3543
3,DT,0.6,0.4322,0.0035,5315
4,DT,0.8,0.4372,0.0029,7087
5,DT,1.0,0.4449,NaN,8859
6,KNN,0.1,0.4155,0.0103,885
7,KNN,0.2,0.4310,0.0050,1771
8,KNN,0.4,0.4434,0.0068,3543
9,KNN,0.6,0.4463,0.0002,5315


In [15]:
if RECOMPUTE:
    # Where three structurally different learners agree, are they right?
    trio = ["DT (τ)", "KNN (τ)", "Rules (M4)"]
    P3 = np.stack([PRED[n] for n in trio])
    agree = (P3 == P3[0]).all(0)
    ag = []
    for c in CLASSES + ["all"]:
        m = agree & ((P3[0] == c) if c != "all" else True)
        ag.append((c, int(m.sum()), m.mean(), (P3[0][m] == yte[m]).mean(), (yte == c).mean() if c != "all" else np.nan))
    ag = pd.DataFrame(ag, columns=["agreed class", "test sellers", "coverage", "precision", "base rate"])
    ag.to_csv(ps.TAB_DIR / "m5_agreement.csv", index=False)
    print("accuracy where they disagree (KNN):", round((P3[1][~agree] == yte[~agree]).mean(), 3))
    print(pd.DataFrame([[accuracy_score(PRED[a], PRED[b]) for b in trio] for a in trio], trio, trio).round(3))
else:
    ag = saved("m5_agreement")
ag.round(3)

,agreed class,test sellers,coverage,precision,base rate
0,low,109,0.039,0.394,0.186
1,medium,775,0.280,0.680,0.547
2,high,563,0.203,0.480,0.266
3,all,1447,0.523,0.581,NaN


**Reading.**
- **Ordinality:** low↔high confusions are 12–19% of the errors against 17–22% expected by chance with the same marginals, so models capture the order, but weakly. An explicit ordinal decomposition (Frank & Hall) adds nothing (QWK 0.235 vs 0.233).
- **Ceiling:** tuned KNN equals the ensembles; the learning curves are flattening but still rising slowly. Better attributes would help more than more sellers.
- **Agreement:** the three learners agree on 52% of the test sellers, where precision is 0.58 (0.68 on `medium`, 0.48 on `high`, 0.39 on `low`).

## 6. Cross-module: seller size, dealer type and generalization of the rules

In [16]:
# (a) Does "small sellers are rated higher" hold inside every segment, or only between segments?
def size_row(name, d):
    small, large = d.n_listings <= d.n_listings.quantile(1 / 3), d.n_listings > d.n_listings.quantile(2 / 3)
    r, p = stats.spearmanr(d.n_listings, as_rank(d[TARGET]))
    return {"segment": name, "n": len(d), "rho(size, rating)": r, "p": p,
            "high % small": (d[TARGET][small] == "high").mean() * 100, "high % large": (d[TARGET][large] == "high").mean() * 100,
            "low % small": (d[TARGET][small] == "low").mean() * 100, "low % large": (d[TARGET][large] == "low").mean() * 100}

simp = pd.DataFrame([size_row("all", df)] + [size_row(f"C{k}", df[clusters.kmeans4 == k]) for k in range(4)]
                    + [size_row(s_, df[df.franchise_status == s_]) for s_ in ["franchise", "independent"]]).set_index("segment")
simp.to_csv(ps.TAB_DIR / "m5_size_effect.csv")
simp.round(3)

,n,"rho(size, rating)",p,high % small,high % large,low % small,low % large
segment,,,,,,,
all,13843,-0.264,0.0,40.641,12.711,13.439,24.794
C0,5069,-0.249,0.0,33.764,8.007,16.794,28.766
C1,3658,-0.309,0.0,41.801,10.049,10.932,22.570
C2,1282,-0.207,0.0,48.252,21.412,16.084,22.353
C3,3834,-0.209,0.0,46.439,22.686,12.424,20.639
franchise,6999,-0.240,0.0,33.461,8.703,15.434,26.842
independent,6844,-0.254,0.0,46.453,17.952,12.035,21.831


In [17]:
# (b) Dealer type after conditioning on size
t3 = pd.qcut(df.n_listings, 3, labels=["small", "mid", "large"])
fx = df.assign(size=t3).groupby(["size", "franchise_status"], observed=True)[TARGET].agg(
    n="size", high=lambda s: (s == "high").mean() * 100, low=lambda s: (s == "low").mean() * 100).unstack().round(1)
fx.to_csv(ps.TAB_DIR / "m5_size_by_dealer.csv")
print(fx)

# Mantel-Haenszel: odds ratio of 'high' for franchise vs independent, crude and pooled over size deciles
def table(d):
    f_, h_ = d.franchise_status.eq("franchise"), d[TARGET].eq("high")
    return (f_ & h_).sum(), (f_ & ~h_).sum(), (~f_ & h_).sum(), (~f_ & ~h_).sum()
a_, b_, c_, d_ = table(df)
strata = [table(g) for _, g in df.groupby(pd.qcut(df.n_listings, 10), observed=True)]
MH = np.array(strata, float)
nT = MH.sum(1)
or_mh = (MH[:, 0] * MH[:, 3] / nT).sum() / (MH[:, 1] * MH[:, 2] / nT).sum()
P_, Q_ = (MH[:, 0] + MH[:, 3]) / nT, (MH[:, 1] + MH[:, 2]) / nT           # Robins-Breslow-Greenland variance of log OR_MH
R_, S_ = MH[:, 0] * MH[:, 3] / nT, MH[:, 1] * MH[:, 2] / nT
se = np.sqrt((P_ * R_).sum() / (2 * R_.sum() ** 2) + (P_ * S_ + Q_ * R_).sum() / (2 * R_.sum() * S_.sum()) + (Q_ * S_).sum() / (2 * S_.sum() ** 2))
crude, se_c = a_ * d_ / (b_ * c_), np.sqrt(1 / a_ + 1 / b_ + 1 / c_ + 1 / d_)
mh = pd.DataFrame({"odds ratio": [crude, or_mh], "lo": [crude * np.exp(-1.96 * se_c), or_mh * np.exp(-1.96 * se)],
                   "hi": [crude * np.exp(1.96 * se_c), or_mh * np.exp(1.96 * se)]}, index=["crude", "adjusted for size decile (MH)"])
mh.to_csv(ps.TAB_DIR / "m5_mantel_haenszel.csv")
print(mh.round(3))

if RECOMPUTE:
    # (c) Do the rating rules generalize? Lift of A -> rating=c mined on the training split, re-measured on the test split
    Yte = np.stack([(yte == c) for c in CLASSES], 1).astype(float)
    cnt_te = Mte.T.astype(float) @ Yte
    n_te = cnt_te.sum(1)
    R = pd.concat([pd.DataFrame({"rating": c, "n_train": rb.cnt_.sum(1), "lift_tr": rb.lift_[:, j], "n_test": n_te,
                                 "lift_te": cnt_te[:, j] / np.maximum(n_te, 1) / Yte[:, j].mean()}) for j, c in enumerate(CLASSES)], ignore_index=True)
    R = R[(R.n_test >= 5) & (R.lift_tr > 1)]
    R["support"] = pd.cut(R.n_train / len(dtr), [0, .01, .02, .05, 1], labels=["0.5-1%", "1-2%", "2-5%", ">5%"])
    top = R.sort_values("lift_tr", ascending=False).groupby(["rating", "support"], observed=True).head(50)
    gen = top.groupby(["rating", "support"], observed=True).agg(rules=("lift_tr", "size"), train_lift=("lift_tr", "mean"), test_lift=("lift_te", "mean"),
                                                               still_positive=("lift_te", lambda s: (s > 1).mean()))
    gen["shrinkage"] = 1 - (gen.test_lift - 1) / (gen.train_lift - 1)
    gen.to_csv(ps.TAB_DIR / "m5_rule_generalization.csv")
else:
    gen = saved("m5_rule_generalization", index_col=[0, 1])
gen.round(3)

                         n                  high                   low            
franchise_status franchise independent franchise independent franchise independent
size                                                                              
small                  336        4285      41.7        40.6      16.1        13.2
mid                   2496        2116      30.0        22.4      15.9        19.7
large                 4167         443      13.2         7.9      24.3        29.8
                               odds ratio     lo     hi
crude                               0.531  0.492  0.573
adjusted for size decile (MH)       1.734  1.542  1.950


rules  train_lift  test_lift  still_positive  shrinkage
rating support                                                         
high   0.5-1%      50       2.136      1.935            0.98      0.177
       1-2%        50       2.035      1.943            1.00      0.089
       2-5%        50       1.953      1.924            1.00      0.031
       >5%         50       1.663      1.594            1.00      0.103
low    0.5-1%      50       2.749      2.254            1.00      0.283
       1-2%        50       2.377      2.147            1.00      0.168
       2-5%        50       2.133      1.964            1.00      0.149
       >5%         50       1.831      1.750            1.00      0.098
medium 0.5-1%      50       1.457      1.235            0.84      0.485
       1-2%        50       1.389      1.260            0.98      0.333
       2-5%        50       1.321      1.213            1.00      0.337
       >5%         50       1.243      1.170            1.00      0.299

In [18]:
fig, axes = plt.subplots(1, 3, figsize=(ps.FULL_W, 2.4))
ax = axes[0]
s_ = simp.loc[["C0", "C1", "C2", "C3"]]
for i, (col, colr, lab) in enumerate([("high % small", "#104281", "smallest third"), ("high % large", "#86b6ef", "largest third")]):
    ax.bar(np.arange(4) + (i - .5) * .38, s_[col], .34, color=colr, label=lab)
    for x, v in enumerate(s_[col]):
        ax.text(x + (i - .5) * .38, v + .8, f"{v:.0f}", ha="center", fontsize=6.5, color=ps.INK_2)
ax.set_xticks(range(4)); ax.set_xticklabels(s_.index); ax.set_ylabel("% rated high"); ax.set_ylim(0, s_["high % small"].max() * 1.45)
ax.set_title("Size effect inside each segment"); ax.legend(frameon=False, fontsize=6.5, loc="upper left"); ax.grid(axis="x", visible=False)
ax = axes[1]
for i, n in enumerate(["DT", "KNN"]):
    q = lc_tab[lc_tab.model == n]
    ax.errorbar(q.n, q["mean"], yerr=q["std"].fillna(0), marker="os"[i], markersize=4, color=ps.CATEGORICAL[i], linestyle=["-", "--"][i],
                label=NICE[n], capsize=2, linewidth=1.5)
ax.axhline(CEIL, color=ps.INK_2, linewidth=0.8, linestyle=":")
ax.text(lc_tab.n.max(), CEIL + 0.001, "best ensemble", fontsize=6.5, color=ps.INK_2, ha="right", va="bottom")
ax.set_xlabel("training sellers"); ax.set_ylabel("CV macro-F1"); ax.set_title("Learning curves"); ax.legend(frameon=False, loc="lower right")
ax = axes[2]
for c in CLASSES:
    q = gen.loc[c]
    ax.plot(q.train_lift, q.test_lift, marker={"low": "o", "medium": "s", "high": "D"}[c], color=ps.CLASS_COLORS[c], label=c, linewidth=1.2, markersize=4)
hi = gen.train_lift.max() * 1.05
ax.plot([0.9, hi], [0.9, hi], color=ps.NEUTRAL, linestyle=":", linewidth=1)
ax.set_xlim(0.9, hi); ax.set_ylim(0.9, hi); ax.set_xlabel("mean lift on train (top-50 rules)"); ax.set_ylabel("mean lift on test")
ax.set_title("Rating rules: train vs test lift"); ax.legend(frameon=False, title="rating", title_fontsize=6.5, loc="upper left")
fig.tight_layout()
ps.save(fig, "m5_cross_module")

**Reading.**
- **The size effect is universal:** in every K-Means segment and dealer type, small sellers are rated `high` 2–4 times as often as large ones.
- **The dealer effect is a Simpson's paradox:** crude odds ratio of `high` for franchises 0.53; adjusted for size deciles (Mantel–Haenszel) 1.73. Independents look better because they are small.
- **Rules:** the best rating rules keep a positive association on the test set. Lift shrinks most at low support (e.g. `low` at 0.5–1%: 2.75 → 2.25): part of an extreme lift is selection.

## 7. Binning sensitivity of pattern mining (Module 4)

In [19]:
if RECOMPUTE:
    def bins_eval(bins, strat):
        enc = fit_encoder(df, bins, strat)
        Bx = encode_items(df, enc)
        fi = apriori(Bx, min_support=0.05, use_colnames=True, low_memory=True)
        ar = association_rules(fi[["support", "itemsets"]], num_itemsets=len(Bx), metric="confidence", min_threshold=0.6,
                               return_metrics=["support", "confidence", "lift"])
        f = []
        for a, b in FOLDS:
            m = RuleClassifier(0.01, 3, bins, strat).fit(dtr.iloc[a], TARGET, CLASSES)
            f.append(f1_score(ytr[b], m.predict(dtr.iloc[b]), average="macro"))
        return {"binning": f"{bins} equal-{'frequency' if strat == 'quantile' else 'width'}", "items": Bx.shape[1], "frequent itemsets": len(fi),
                "rules": len(ar), "median lift": ar.lift.median(), "max lift": ar.lift.max(),
                "largest numeric-bin support": encode_items(df, {k: v for k, v in enc.items() if k != "vocab"}).filter(regex="^(size|price|mileage|dom|hp|span|cab)=").mean().max(),
                "rule classifier CV macro-F1": np.mean(f), "sd": np.std(f)}

    bt = pd.DataFrame(Parallel(n_jobs=N_JOBS, max_nbytes=None)(delayed(bins_eval)(b, s) for b, s in
                      [(2, "quantile"), (3, "quantile"), (4, "quantile"), (5, "quantile"), (3, "width"), (5, "width")]))
    bt.to_csv(ps.TAB_DIR / "m5_binning.csv", index=False)
else:
    bt = saved("m5_binning")
bt.round(3)

,binning,items,frequent itemsets,rules,median lift,max lift,largest numeric-bin support,rule classifier CV macro-F1,sd
0,2 equal-frequency,40,13603,153578,2.048,4.878,0.504,0.363,0.013
1,3 equal-frequency,47,4977,24787,2.255,6.411,0.347,0.390,0.008
2,4 equal-frequency,54,2706,7395,2.104,5.194,0.264,0.420,0.010
3,5 equal-frequency,61,1591,3340,1.978,4.977,0.211,0.403,0.012
4,3 equal-width,43,1270,6180,1.869,3.461,1.000,0.264,0.010
5,5 equal-width,56,2028,12108,1.842,2.829,0.999,0.289,0.010


**Reading.**
- Equal-width bins fail: one bin holds almost all sellers (support up to 1.0) and the classifier drops to 0.26–0.29. This confirms the Module 4 choice of equal-frequency bins.
- Fewer bins give many more, less specific rules. In the rule classifier (s = 0.01, L = 3) quartiles do better than tertiles (0.420 vs 0.390). Tertiles are kept for the descriptive part because *low/mid/high* is readable; this is a possible improvement of the classifier that was not tuned further.

## 8. Regression: predictor sets and target scale

In [20]:
if RECOMPUTE:
    KF = KFold(5, shuffle=True, random_state=SEED)
    P4 = ["mileage_median", "horsepower_median", "width_median", "pct_is_new"]
    PALL = [c for c in NUM if c != "price_median"]
    REG = {"OLS": (LinearRegression(), {"reg__fit_intercept": [True]}),
           "DT": (DecisionTreeRegressor(random_state=SEED), {"reg__max_depth": [8, 10, 12, 15], "reg__min_samples_leaf": [5, 10, 20, 50]}),
           "KNN": (KNeighborsRegressor(p=1, weights="distance"), {"reg__n_neighbors": [5, 10, 15, 20, 30]})}
    ptr_, pte_ = dtr.price_median.to_numpy(), dte.price_median.to_numpy()
    reg_rows, REGB = [], {}
    for pset, cols in [("1 (mileage)", ["mileage_median"]), ("4 (Module 3)", P4), ("18 (all numeric)", PALL)]:
        for tname in ["raw", "log1p"]:
            y_ = ptr_ if tname == "raw" else np.log1p(ptr_)
            for mname, (base, g) in REG.items():
                inner = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()), ("reg", base)])
                s = GridSearchCV(inner, g, cv=KF, scoring="neg_mean_squared_error", n_jobs=N_JOBS, error_score="raise").fit(dtr[cols], y_)
                oof = cross_val_predict(s.best_estimator_, dtr[cols], y_, cv=KF, n_jobs=N_JOBS)
                usd = oof if tname == "raw" else np.expm1(oof)
                REGB[(pset, tname, mname)] = (s.best_estimator_, cols, y_)
                reg_rows.append({"predictors": pset, "target": tname, "model": mname, "CV R2 (log)": r2_score(np.log1p(ptr_), np.log1p(np.maximum(usd, 1))),
                                 "CV R2 ($)": r2_score(ptr_, usd), "median abs. % error": np.median(np.abs(usd - ptr_) / ptr_) * 100,
                                 "negative predictions": int((usd < 0).sum()), "params": str({k[5:]: v for k, v in s.best_params_.items() if "intercept" not in k})})
    rg = pd.DataFrame(reg_rows)
    rg.to_csv(ps.TAB_DIR / "m5_regression_ablation.csv", index=False)
else:
    rg = saved("m5_regression_ablation")
rg.round(3)

,predictors,target,model,CV R2 (log),CV R2 ($),median abs. % error,negative predictions,params
0,1 (mileage),raw,OLS,-1.059,0.465,23.384,107,{}
1,1 (mileage),raw,DT,0.731,0.497,20.902,0,"{'max_depth': 12, 'min_samples_leaf': 50}"
2,1 (mileage),raw,KNN,0.692,0.416,21.495,0,{'n_neighbors': 30}
3,1 (mileage),log1p,OLS,0.740,0.474,19.446,0,{}
4,1 (mileage),log1p,DT,0.744,0.494,19.372,0,"{'max_depth': 8, 'min_samples_leaf': 50}"
5,1 (mileage),log1p,KNN,0.703,0.433,20.187,0,{'n_neighbors': 30}
6,4 (Module 3),raw,OLS,-1.870,0.695,17.173,170,{}
7,4 (Module 3),raw,DT,0.898,0.849,10.479,0,"{'max_depth': 12, 'min_samples_leaf': 10}"
8,4 (Module 3),raw,KNN,0.910,0.874,9.682,0,{'n_neighbors': 10}
9,4 (Module 3),log1p,OLS,0.880,0.763,12.950,0,{}


In [21]:
if RECOMPUTE:
    fin = []
    for pset in ["4 (Module 3)", "18 (all numeric)"]:
        for mname in REG:
            est, cols, y_ = REGB[(pset, "log1p", mname)]
            est = clone(est).fit(dtr[cols], y_)
            p = est.predict(dte[cols])
            fin.append({"predictors": pset, "model": mname, "train MSE": mean_squared_error(y_, est.predict(dtr[cols])), "test MSE": mean_squared_error(np.log1p(pte_), p),
                        "test R2": r2_score(np.log1p(pte_), p), "test R2 ($)": r2_score(pte_, np.expm1(p)),
                        "median abs. % error": np.median(np.abs(np.expm1(p) - pte_) / pte_) * 100})
    fin = pd.DataFrame(fin)
    fin.to_csv(ps.TAB_DIR / "m5_regression_test.csv", index=False)
else:
    fin = saved("m5_regression_test")
fin.round(4)

,predictors,model,train MSE,test MSE,test R2,test R2 ($),median abs. % error
0,4 (Module 3),OLS,0.0462,0.0479,0.8758,0.7634,13.3844
1,4 (Module 3),DT,0.0284,0.0376,0.9026,0.8691,10.2576
2,4 (Module 3),KNN,0.0000,0.0337,0.9128,0.8735,10.0075
3,18 (all numeric),OLS,0.0396,0.0402,0.8959,0.8212,11.7356
4,18 (all numeric),DT,0.0231,0.0346,0.9103,0.8716,9.7236
5,18 (all numeric),KNN,0.0000,0.0354,0.9083,0.8416,9.2564


**Reading.**
- On the raw scale OLS predicts negative prices for 107–206 sellers; the log target is essential for the linear models and irrelevant for DT/KNN.
- 18 predictors improve OLS (0.880 → 0.896) and DT (0.903 → 0.910) but not KNN. On the test set nothing beats KNN with four predictors (R² 0.913).